# Purpose

Prove the persona certification contract authority is bounded, deterministic, and complete.

## Lemma Mapping

- L64 — M365 Persona Certification v1

## Invariant Support

- invariants/lemmas/L64_m365_persona_certification_v1.yaml

## Expected Results

- The persona certification contract validates with four ordered phases.
- Governance rules are complete.
- All 39 personas pass four-phase certification.
- Coverage partition: 4 registry-backed + 35 contract-only = 39 total.

In [ ]:
from pathlib import Path
import yaml

repo_root = Path.cwd()
contract = yaml.safe_load(
    (repo_root / 'registry' / 'persona_certification_v1.yaml').read_text(encoding='utf-8')
)

assert contract['contract']['id'] == 'persona-certification'
assert contract['contract']['status'] == 'contract-defined'

phases = contract['certification_phases']
assert len(phases) == 4
assert [p['order'] for p in phases] == [1, 2, 3, 4]

rules = contract['governance_rules']
assert set(rules.keys()) == {
    'fail_closed', 'audit_completeness', 'bounded_claims',
    'determinism', 'coverage_partition',
}

In [ ]:
registry = yaml.safe_load(
    (repo_root / 'registry' / 'persona_registry_v2.yaml').read_text(encoding='utf-8')
)
personas = registry['personas']
kpis = contract['kpis']

assert kpis['total_personas'] == len(personas)

REQUIRED_FIELDS = {
    'persona_id', 'display_name', 'slug', 'title', 'department', 'status',
    'coverage_status', 'risk_tier', 'approval_profile', 'approval_owner',
    'escalation_owner', 'manager', 'canonical_agent', 'external_presence_policy',
    'responsibilities', 'capability_families', 'workload_families',
    'allowed_domains', 'allowed_actions', 'action_count', 'aliases',
}

rb_count = 0
co_count = 0
for pid, p in personas.items():
    assert REQUIRED_FIELDS.issubset(set(p.keys())), f'{pid} missing fields'
    if p['coverage_status'] == 'registry-backed':
        assert p['action_count'] > 0, f'{pid} registry-backed but 0 actions'
        rb_count += 1
    else:
        assert p['action_count'] == 0, f'{pid} contract-only but has actions'
        co_count += 1

assert rb_count == kpis['registry_backed_personas']
assert co_count == kpis['contract_only_personas']
assert rb_count + co_count == kpis['total_personas']